# 2. Data Cleaning


In [1]:
import pandas as pd
import numpy as np

## Step 1. Load and prepare the columns
Load with `sep=";"` and `decimal=","` so `volume` and `wm_vsola` are read as numbers (European number format).


In [2]:
raw_df = pd.read_csv("../../data/anonymized_logistics_picking_data.csv", sep=";", decimal=",")

print(raw_df.shape)
raw_df.dtypes

(217963, 18)


location           str
volume         float64
picker             str
wm_vsola       float64
area               str
warehouse          str
calday             str
calweek          int64
calmonth           str
calyear            str
collli_id          str
start_time         str
finish_time        str
wm_srsrc       float64
wm_vltyp         int64
product_id         str
wm_qdocno      float64
wm_tapos         int64
dtype: object

### 1.1 Drop columns without information

These columns have only one value or are (almost) entirely empty, so they cannot explain any variation.

In [3]:
drop_cols = ['area', 'warehouse', 'wm_vltyp', 'calmonth', 'calyear', 'wm_srsrc', 'wm_qdocno']

df = raw_df.drop(columns=drop_cols)
print(df.shape)

(217963, 11)


### 1.2 Rename columns

In [4]:
df = df.rename(columns={
    'collli_id': 'colli_id',          # fix the typo (three l's)
    'wm_vsola': 'quantity',           # number of cases picked
    'wm_tapos': 'item_position'       # line position within a task (1 = first, 2/3 = continuation)
})
df.columns.tolist()

['location',
 'volume',
 'picker',
 'quantity',
 'calday',
 'calweek',
 'colli_id',
 'start_time',
 'finish_time',
 'product_id',
 'item_position']

### 1.3 Fix data types

- `finish_time` → datetime. It is complete and is the only timestamp used for metrics.
- `start_time` → datetime, kept for reference only.
- `quantity` → integer. Every value is a whole number.
- `calday` and `calweek` are kept for now; they are replaced by `shift_date`.

In [5]:
df['finish_time'] = pd.to_datetime(df['finish_time'])
df['start_time'] = pd.to_datetime(df['start_time'])

# Safety check before converting: every quantity must be a whole number
assert (df['quantity'] % 1 == 0).all(), "Non-whole quantities found" # assert stops the notebook with "Non-whole quantities found" if any value fails
df['quantity'] = df['quantity'].astype('int64')

df.dtypes

location                    str
volume                  float64
picker                      str
quantity                  int64
calday                      str
calweek                   int64
colli_id                    str
start_time       datetime64[us]
finish_time      datetime64[us]
product_id                  str
item_position             int64
dtype: object

### 1.4 Check the result

In [6]:
print('Rows:', len(df))
print('Columns:', df.shape[1])
print()
print(df.isna().sum())

Rows: 217963
Columns: 11

location          0
volume            0
picker            0
quantity          0
calday            0
calweek           0
colli_id          0
start_time       94
finish_time       0
product_id        0
item_position     0
dtype: int64


In [7]:
cleaning_log = [{'step': '1. Load and prepare columns', 'rows': len(df), 'note': 'Dropped 7 columns, renamed, fixed types'}]
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"


## 2. Exclude the placeholder colli

`colli_6599` is not a real store order: 727 lines, 36 pickers, a span of about 19 days, and most of its activity outside the night shift. It most likely records non-order activity under one catch-all ID, so it is excluded from all picking analysis.

In [8]:

# check before removing
PLACEHOLDER_COLLI = 'colli_6599'

is_placeholder = df['colli_id'] == PLACEHOLDER_COLLI
placeholder = df[is_placeholder]

print('Rows in placeholder:', len(placeholder))
print('Pickers:', placeholder['picker'].nunique())
print('Zero-volume rows:', (placeholder['volume'] == 0).sum())
print('Missing start_time:', placeholder['start_time'].isna().sum())
print('Off-shift picks (04:00–20:59):', placeholder['finish_time'].dt.hour.between(4, 20).sum())

Rows in placeholder: 727
Pickers: 36
Zero-volume rows: 192
Missing start_time: 9
Off-shift picks (04:00–20:59): 466


In [9]:
#Remove placeholder rows

rows_before = len(df)
df = df[~is_placeholder].copy()


print('Rows before:', rows_before)
print('Rows removed:', rows_before - len(df))
print('Rows remaining:', len(df))

Rows before: 217963
Rows removed: 727
Rows remaining: 217236


In [10]:
# Chheck the results

print('Placeholder rows left:', (df['colli_id'] == PLACEHOLDER_COLLI).sum())
print('Zero-volume rows left:', (df['volume'] == 0).sum())
print('Missing start_time left:', df['start_time'].isna().sum())
print('Off-shift picks left:', df['finish_time'].dt.hour.between(4, 20).sum())
print('Max quantity:', df['quantity'].max())
print('Pickers left:', df['picker'].nunique())

Placeholder rows left: 0
Zero-volume rows left: 7
Missing start_time left: 85
Off-shift picks left: 4
Max quantity: 486
Pickers left: 32


In [11]:
# Record to cleaning log
cleaning_log.append({'step': '2. Exclude placeholder colli', 'rows': len(df), 'note': 'Removed colli_6599 (727 rows)'})
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)


## 3. Set zero volume to missing

A pick cannot have zero volume. These rows have normal quantities, so the volume was not recorded. Setting it to missing (NaN) keeps the picks in pick counts without pulling volume averages down with false zeros.

Negative volumes (5 rows) are **not** changed here; they are netted into their original pick in 5.

In [12]:
zero_volume = df['volume'] == 0
print('Zero-volume rows:', zero_volume.sum())

df.loc[zero_volume, 'volume'] = np.nan

Zero-volume rows: 7


In [13]:
# check the results
print('Missing volume:', df['volume'].isna().sum())
print('Zero volume left:', (df['volume'] == 0).sum())


Missing volume: 7
Zero volume left: 0


In [14]:
cleaning_log.append({'step': '3. Zero volume to NaN', 'rows': len(df), 'note': '7 zero volumes set to missing'})
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing


## 4. Create the shift calendar

The night shift runs from about 21:00 to 03:00, so `calday` (based on `finish_time`) splits every shift at midnight. `shift_date` assigns each pick to the **evening the shift starts**: picks before noon belong to the previous evening's shift.

From `shift_date` we derive:
- `shift_weekday` and `shift_week` (replacing `calday` and `calweek`)
- `holiday_period`: shifts 5–10 April due to Easter
- `partial_shift`: the first and last shifts, which are only half in the data

 4.1 Create a new column for the shift date.

In [15]:


#.dt.date turns the column into plain Python date objects, which pandas stores as text-like objects.
# .dt.normalize() keeps it as a proper datetime column, which works better for sorting,

SHIFT_CUTOFF_HOURS = 12   # picks before 12:00 belong to the previous evening's shift

df['shift_date'] = (df['finish_time'] - pd.Timedelta(hours=SHIFT_CUTOFF_HOURS)).dt.normalize()

print('Shifts:', df['shift_date'].nunique())
print('First shift:', df['shift_date'].min().date(), '| Last shift:', df['shift_date'].max().date())

Shifts: 21
First shift: 2023-03-31 | Last shift: 2023-04-20


 4.2 Weeday and Week from the shift

In [16]:


df['shift_weekday'] = df['shift_date'].dt.day_name()
df['shift_week'] = df['shift_date'].dt.isocalendar().week.astype('int64')

4.3 Holiday preriod and partial shifts

In [17]:
# Easter period (5–10 April): low and peak shifts that would distort normal-day averages,
# so they are flagged and analysed separately
HOLIDAY_SHIFTS = pd.date_range('2023-04-05', '2023-04-10')

# First and last shifts are only half in the data (it starts and ends at midnight),
# so they are flagged and excluded from shift-level comparisons
PARTIAL_SHIFTS = pd.to_datetime(['2023-03-31', '2023-04-20'])

df['holiday_period'] = df['shift_date'].isin(HOLIDAY_SHIFTS)
df['partial_shift'] = df['shift_date'].isin(PARTIAL_SHIFTS)

4.4 Drop 'calday' and 'calweek'

In [18]:


df = df.drop(columns=['calday', 'calweek'])
df.columns.tolist()

['location',
 'volume',
 'picker',
 'quantity',
 'colli_id',
 'start_time',
 'finish_time',
 'product_id',
 'item_position',
 'shift_date',
 'shift_weekday',
 'shift_week',
 'holiday_period',
 'partial_shift']

In [19]:
# Check the results
shifts = df.groupby('shift_date').agg(
    picks=('product_id', 'count'),
    first_pick=('finish_time', 'min'),
    last_pick=('finish_time', 'max'),
    weekday=('shift_weekday', 'first'),
    holiday=('holiday_period', 'first'),
    partial=('partial_shift', 'first')
)
shifts

,picks,first_pick,last_pick,weekday,holiday,partial
shift_date,,,,,,
2023-03-31,6244,2023-04-01 00:00:02,2023-04-01 02:52:34,Friday,False,True
2023-04-01,11876,2023-04-01 21:12:54,2023-04-02 03:31:33,Saturday,False,False
2023-04-02,9640,2023-04-02 21:05:04,2023-04-03 03:54:16,Sunday,False,False
2023-04-03,10985,2023-04-03 21:12:53,2023-04-04 02:21:38,Monday,False,False
2023-04-04,12608,2023-04-04 21:08:54,2023-04-05 03:40:54,Tuesday,False,False
2023-04-05,5485,2023-04-05 21:13:40,2023-04-06 02:23:39,Wednesday,True,False
2023-04-06,5367,2023-04-06 21:12:08,2023-04-07 02:00:55,Thursday,True,False
2023-04-07,15450,2023-04-07 21:07:00,2023-04-08 03:49:43,Friday,True,False
2023-04-08,5790,2023-04-08 21:03:26,2023-04-09 02:13:04,Saturday,True,False


In [20]:
# Add to Cleaning Log

print('Holiday shifts:', shifts['holiday'].sum())
print('Partial shifts:', shifts['partial'].sum())
print('Rows:', len(df), '| Columns:', df.shape[1])

cleaning_log.append({'step': '4. Shift calendar', 'rows': len(df),
                     'note': 'Added shift_date, weekday, week, holiday and partial flags; dropped calday, calweek'})
pd.DataFrame(cleaning_log)

Holiday shifts: 6
Partial shifts: 2
Rows: 217236 | Columns: 14


,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."


## Step 5. Merge lines into picks

**Pick definition:** one pick = one product picked into one colli during one visit.

Lines of the same product in the same colli are merged into one pick. This covers:
- the 5 negative-volume corrections
- the fast repeated `item_position = 1` lines
- the continuation lines `item_position = 2/3`

For each pick: quantities and volumes are **summed**, the pick keeps the **finish time of its last line**, and flags show whether it had continuation or repeat lines.

5.1 Check that same-product lines really are one visit

Before merging, prove that the lines being merged really are close together in time and done by the same picker. Otherwise you'd be merging two separate trips.

In [21]:
lines = df.sort_values(['colli_id', 'product_id', 'finish_time']).copy()

lines['gap_same_product_sec'] = (
    lines.groupby(['colli_id', 'product_id'])['finish_time'].diff().dt.total_seconds()
)

gaps = lines['gap_same_product_sec'].dropna()
print('Lines to merge:', len(gaps))
print(gaps.describe(percentiles=[0.5, 0.9, 0.99]).round(0))
print('Gaps over 2 minutes:', (gaps > 120).sum())
print('Pickers per colli-product:', lines.groupby(['colli_id', 'product_id'])['picker'].nunique().max())


# 11,142 lines to merge (8,903 continuation lines + 2,239 fast repeats).
# A median gap of 4 seconds, with 99% within 49 seconds.
# Only 15 gaps over 2 minutes, all within the same colli and done by the same picker, so they're the same task 
#                       (for example, waiting briefly to finish the pick).
# 1 picker per colli-product.

# That confirms they're one visit, so we can merge safely.

Lines to merge: 11142
count    11142.0
mean         8.0
std         18.0
min          1.0
50%          4.0
90%         17.0
99%         49.0
max       1342.0
Name: gap_same_product_sec, dtype: float64
Gaps over 2 minutes: 15
Pickers per colli-product: 1


5.2 Merge line into picks

In [22]:
key = ['colli_id', 'product_id']
grouped = lines.groupby(key)

picks = grouped.agg(
    picker=('picker', 'first'),
    location=('location', 'first'),
    first_finish_time=('finish_time', 'min'),
    finish_time=('finish_time', 'max'),          # keep the last line's finish time
    start_time=('start_time', 'min'),            # for reference only
    quantity=('quantity', 'sum'),                # sum the cases
    n_lines=('item_position', 'size'),
    max_item_position=('item_position', 'max'),
    shift_date=('shift_date', 'last'),
    shift_weekday=('shift_weekday', 'last'),
    shift_week=('shift_week', 'last'),
    holiday_period=('holiday_period', 'last'),
    partial_shift=('partial_shift', 'last'),
).reset_index()

# Volume: sum separately with min_count=1, so a pick whose volume is entirely missing stays NaN, not 0
picks['volume'] = grouped['volume'].sum(min_count=1).values

picks.sort_values(['colli_id', 'picker', 'finish_time']).head()

,colli_id,product_id,picker,location,first_finish_time,finish_time,start_time,quantity,n_lines,max_item_position,shift_date,shift_weekday,shift_week,holiday_period,partial_shift,volume
4,colli_0,product_111,picker_0,10-01-014-1,2023-04-01 01:16:15,2023-04-01 01:16:15,2023-04-01 01:15:14,5,1,1,2023-03-31,Friday,13,False,True,0.022
29,colli_0,product_90,picker_0,10-01-015-1,2023-04-01 01:16:20,2023-04-01 01:16:20,2023-04-01 01:16:08,14,1,1,2023-03-31,Friday,13,False,True,0.013
2,colli_0,product_107,picker_0,10-01-016-1,2023-04-01 01:16:24,2023-04-01 01:16:24,2023-04-01 01:16:14,8,1,1,2023-03-31,Friday,13,False,True,0.014
26,colli_0,product_85,picker_0,10-01-018-1,2023-04-01 01:16:40,2023-04-01 01:16:40,2023-04-01 01:16:18,9,1,1,2023-03-31,Friday,13,False,True,0.011
8,colli_0,product_140,picker_0,10-01-020-1,2023-04-01 01:16:44,2023-04-01 01:16:44,2023-04-01 01:16:33,12,1,1,2023-03-31,Friday,13,False,True,0.044


In [23]:
# look how picker 0 collct the colli 0 by sorting finish time by ascending order
picker_0_colli_0 = picks[(picks['picker'] =='picker_0') & (picks['colli_id'] == 'colli_0')].sort_values('finish_time')

picker_0_colli_0

,colli_id,product_id,picker,location,first_finish_time,finish_time,start_time,quantity,n_lines,max_item_position,shift_date,shift_weekday,shift_week,holiday_period,partial_shift,volume
4,colli_0,product_111,picker_0,10-01-014-1,2023-04-01 01:16:15,2023-04-01 01:16:15,2023-04-01 01:15:14,5,1,1,2023-03-31,Friday,13,False,True,0.022
29,colli_0,product_90,picker_0,10-01-015-1,2023-04-01 01:16:20,2023-04-01 01:16:20,2023-04-01 01:16:08,14,1,1,2023-03-31,Friday,13,False,True,0.013
2,colli_0,product_107,picker_0,10-01-016-1,2023-04-01 01:16:24,2023-04-01 01:16:24,2023-04-01 01:16:14,8,1,1,2023-03-31,Friday,13,False,True,0.014
26,colli_0,product_85,picker_0,10-01-018-1,2023-04-01 01:16:40,2023-04-01 01:16:40,2023-04-01 01:16:18,9,1,1,2023-03-31,Friday,13,False,True,0.011
8,colli_0,product_140,picker_0,10-01-020-1,2023-04-01 01:16:44,2023-04-01 01:16:44,2023-04-01 01:16:33,12,1,1,2023-03-31,Friday,13,False,True,0.044
22,colli_0,product_63,picker_0,10-01-021-1,2023-04-01 01:16:57,2023-04-01 01:16:57,2023-04-01 01:16:39,12,1,1,2023-03-31,Friday,13,False,True,0.043
23,colli_0,product_71,picker_0,10-01-023-1,2023-04-01 01:17:21,2023-04-01 01:17:21,2023-04-01 01:16:51,18,1,1,2023-03-31,Friday,13,False,True,0.075
30,colli_0,product_95,picker_0,10-01-026-1,2023-04-01 01:17:28,2023-04-01 01:17:28,2023-04-01 01:17:15,12,1,1,2023-03-31,Friday,13,False,True,0.012
27,colli_0,product_86,picker_0,10-01-030-1,2023-04-01 01:17:48,2023-04-01 01:17:48,2023-04-01 01:17:22,10,1,1,2023-03-31,Friday,13,False,True,0.017
19,colli_0,product_25,picker_0,10-01-041-1,2023-04-01 01:18:11,2023-04-01 01:18:11,2023-04-01 01:17:42,7,1,1,2023-03-31,Friday,13,False,True,0.022


5.3 Flags

A continuation is the system's second item line (item_position = 2), probably the next batch. A repeat is the same product confirmed again on item_position = 1, which might be a double scan. Keeping them apart means split-rate analysis can use has_continuation without mixing in the repeats.

In [24]:
picks['has_continuation'] = picks['max_item_position'] >= 2                          # probably the next batch
picks['has_repeat'] = (picks['n_lines'] > 1) & ~picks['has_continuation']            # fast repeats, double scan
picks = picks.drop(columns='max_item_position')

print('Picks with continuation lines:', picks['has_continuation'].sum())
print('Picks with fast repeats:', picks['has_repeat'].sum())

Picks with continuation lines: 8902
Picks with fast repeats: 1290


In [25]:
# check the results
print('Lines before:', len(lines), '| Picks after:', len(picks), '| Lines merged:', len(lines) - len(picks))

# Cases and volume must be conserved: merging changes the number of rows, not the totals
print('Cases before:', lines['quantity'].sum(), '| after:', picks['quantity'].sum())
print('Volume before:', round(lines['volume'].sum(), 3), '| after:', round(picks['volume'].sum(), 3))

print('Negative volume left:', (picks['volume'] < 0).sum())

Lines before: 217236 | Picks after: 206094 | Lines merged: 11142
Cases before: 1945575 | after: 1945575
Volume before: 4373.205 | after: 4373.205
Negative volume left: 0


In [26]:
# spot check of one pick with multiple lines, to see if the merge worked correctly

picks[(picks['colli_id'] == 'colli_7747') & (picks['product_id'] == 'product_107')].T

,197261
colli_id,colli_7747
product_id,product_107
picker,picker_4
location,10-01-016-1
first_finish_time,2023-04-13 22:56:18
finish_time,2023-04-13 22:56:54
start_time,2023-04-13 22:47:11
quantity,425
n_lines,2
shift_date,2023-04-13 00:00:00


In [27]:
# Add to Cleaning Log
cleaning_log.append({'step': '5. Merge lines into picks', 'rows': len(picks),
                     'note': '11,142 lines merged; cases and volume conserved'})

df = picks   # from here on, df is the pick-level table
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."
4,5. Merge lines into picks,206094,"11,142 lines merged; cases and volume conserved"


## 6. Add analysis fields and flags

All flags are calculated on **picks** (after section 5), not on raw lines.

| Field 
|---|
| `aisle`, `bay` 
| `is_regular_picker` 
| `picker_segment`
| `is_single_line` 
| `is_bulk_line` 
| `is_new_product` 

6.1 Aisle and bay column

In [28]:
# Split the location code (e.g. 10-06-036-1) into aisle and bay.
# Zone (10) and level (1) never change, so only aisle and bay are useful for layout analysis.
loc_parts = df['location'].str.split('-', expand=True)
df['aisle'] = loc_parts[1].astype(int)
df['bay'] = loc_parts[2].astype(int)

df.groupby('aisle').agg(picks=('product_id', 'count'), locations=('location', 'nunique'))

,picks,locations
aisle,,
1,71857,40
2,11104,32
3,15659,32
4,8773,15
5,7426,24
6,91275,41


6.2 Regular pickers and segments

In [29]:
# Pickers below this have too few picks for a fair comparison (natural gap: 36 vs 1,809 picks)
MIN_PICKS_REGULAR = 1000

# Split regular pickers by experience: core pickers work most shifts, occasional ones only a few
# (no picker worked 7 or 8 shifts, so the boundary follows a natural gap)
CORE_MIN_SHIFTS = 9
OCCASIONAL_MAX_SHIFTS = 6

picker_stats = df.groupby('picker').agg(
    picks=('product_id', 'count'),
    shifts=('shift_date', 'nunique')
)

# Assign each picker to a group; 'check' catches anyone who falls between the boundaries
picker_stats['segment'] = np.select(
    [picker_stats['picks'] < MIN_PICKS_REGULAR,
     picker_stats['shifts'] >= CORE_MIN_SHIFTS,
     picker_stats['shifts'] <= OCCASIONAL_MAX_SHIFTS],
    ['non-regular', 'core', 'occasional'],
    default='check'
)

# Non-regular users stay in volume totals but are excluded from time and productivity metrics
df['is_regular_picker'] = df['picker'].map(picker_stats['picks'] >= MIN_PICKS_REGULAR)
df['picker_segment'] = df['picker'].map(picker_stats['segment'])

print(picker_stats['segment'].value_counts())
picker_stats.sort_values('picks').head(6)

segment
core           21
occasional      7
non-regular     4
Name: count, dtype: int64


,picks,shifts,segment
picker,,,
picker_34,6,4,non-regular
picker_31,20,11,non-regular
picker_32,23,9,non-regular
picker_33,36,4,non-regular
picker_27,1809,4,occasional
picker_14,2052,5,occasional


6.3 Single Line Collis

In [30]:
# A colli with only one pick always has a colli duration of 0 seconds,
# so these collis are flagged and excluded from colli-level metrics (but kept as picks)

picks_per_colli = df.groupby('colli_id')['product_id'].transform('count')
df['is_single_line'] = picks_per_colli == 1

print('Single-line collis:', df.loc[df['is_single_line'], 'colli_id'].nunique(), 'of', df['colli_id'].nunique())

Single-line collis: 382 of 8368


6.4 Bulk Lines

In [31]:
# Large-quantity picks are real store orders, so they are kept (not capped),
# but flagged to check whether results change without them
BULK_MIN_CASES = 100

df['is_bulk_line'] = df['quantity'] >= BULK_MIN_CASES
print('Bulk picks:', df['is_bulk_line'].sum())

Bulk picks: 580


6.5 New products

In [32]:
# Products that first appear in the last 2 shifts: too few picks to compare with regular products
NEW_PRODUCTS = ['product_177', 'product_179', 'product_180', 'product_181', 'product_183']

df['is_new_product'] = df['product_id'].isin(NEW_PRODUCTS)
print('Picks of new products:', df['is_new_product'].sum())

Picks of new products: 32


In [33]:
# Check against the data: the last 8 products to appear in the data are the new products, and they have very few picks
product_dates = df.groupby('product_id').agg(
    picks=('colli_id', 'count'),
    first_shift=('shift_date', 'min'),
    last_shift=('shift_date', 'max'),
    shifts=('shift_date', 'nunique')
)
product_dates.sort_values('first_shift').tail(8)

,picks,first_shift,last_shift,shifts
product_id,,,,
product_141,822,2023-04-04,2023-04-13,9
product_162,208,2023-04-14,2023-04-14,1
product_183,2,2023-04-18,2023-04-19,2
product_179,7,2023-04-19,2023-04-20,2
product_170,223,2023-04-19,2023-04-19,1
product_177,11,2023-04-19,2023-04-20,2
product_181,4,2023-04-19,2023-04-20,2
product_180,8,2023-04-19,2023-04-20,2


6.6 Products picked in only one shift

Found while building the new-product flag: some products have plenty of picks, but all from a single night. They are kept in totals but excluded from product comparisons, because one night is not comparable with 20 nights.

In [34]:
# Products whose picks all come from a single shift (e.g. a one-off delivery):
# kept in totals, but excluded from product comparisons because one night is not comparable with 20
product_shifts = df.groupby('product_id')['shift_date'].nunique()
ONE_SHIFT_PRODUCTS = product_shifts[product_shifts == 1].index.tolist()

df['is_one_shift_product'] = df['product_id'].isin(ONE_SHIFT_PRODUCTS)

print('One-shift products:', ONE_SHIFT_PRODUCTS)
print('Picks of one-shift products:', df['is_one_shift_product'].sum())

One-shift products: ['product_162', 'product_170', 'product_182']
Picks of one-shift products: 432


In [35]:
# check the results and add to cleaning log
print('Rows:', len(df), '| Columns:', df.shape[1])

cleaning_log.append({'step': '6. Analysis fields and flags', 'rows': len(df),
                     'note': 'Added aisle, bay, picker flags and segments, single-line, bulk, new-product and one-shift-product flags'})
pd.DataFrame(cleaning_log)

Rows: 206094 | Columns: 25


,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."
4,5. Merge lines into picks,206094,"11,142 lines merged; cases and volume conserved"
5,6. Analysis fields and flags,206094,"Added aisle, bay, picker flags and segments, s..."


## 7. Calculate time per pick

Time per pick = the **finish-to-finish gap** between a picker's consecutive picks **within the same shift**.

- `start_time` is not used: it overlaps the previous pick in 91% of cases.
- Gaps of **15 minutes or more** are breaks (scheduled breaks cluster around 20 minutes at 23:00, 01:00 and 03:00); shorter gaps count as working time, including order changeovers.
- The first pick of each shift has no previous pick, so it has no time.
- Time is calculated for **regular pickers only**: non-regular users make only a few picks per shift, so their gaps say nothing about picking speed.

7.1 Finish-to-finish gaps

In [36]:
# Sort each picker's picks in time order, so each pick can be compared with the one before it
df = df.sort_values(['picker', 'finish_time']).reset_index(drop=True)

# Gap since the picker's previous pick, within the same shift only,
# so the time between two shifts is never counted
df['gap_sec'] = df.groupby(['picker', 'shift_date'])['finish_time'].diff().dt.total_seconds()

print('First pick of a shift (no previous pick):', df['gap_sec'].isna().sum())

First pick of a shift (no previous pick): 361


7.2 Breaks and time per pick

In [37]:
# Gaps of 15 minutes or more are breaks: they fall in the quiet zone between order changeovers
# (up to ~10 min) and scheduled breaks (~20 min), so they are not counted as picking time
BREAK_MIN_SECONDS = 15 * 60

df['is_after_break'] = df['gap_sec'] >= BREAK_MIN_SECONDS

# Time per pick only for regular pickers and only for gaps that are not breaks;
# everything else stays missing, so it is ignored in averages
df['time_per_pick_sec'] = df['gap_sec'].where(~df['is_after_break'] & df['is_regular_picker'])

print('Picks with a time:', df['time_per_pick_sec'].notna().sum())
print('Breaks (regular pickers):', (df['is_after_break'] & df['is_regular_picker']).sum())

Picks with a time: 205001
Breaks (regular pickers): 675


Why keep gap_sec as well? gap_sec is the raw measurement and time_per_pick_sec is the cleaned metric. Keeping both lets anyone check how the metric was derived. It also lets you test a different break threshold later without recalculating anything.

In [38]:
#check the results 
t = df['time_per_pick_sec']

print(t.describe(percentiles=[0.5, 0.9, 0.99]).round(1))
print()
print('Working hours (regular pickers):', round(t.sum() / 3600, 1))
print('Overall picks per hour:', round(t.notna().sum() / (t.sum() / 3600), 1))
print('Time values for non-regular users:', df.loc[~df['is_regular_picker'], 'time_per_pick_sec'].notna().sum())
print('Zero-second picks:', (t == 0).sum())


	# 0 sec pick are 98 - two different collis confirmed at the same second, so two real picks

count    205001.0
mean         23.6
std          44.3
min           0.0
50%          13.0
90%          38.0
99%         253.0
max         874.0
Name: time_per_pick_sec, dtype: float64

Working hours (regular pickers): 1345.4
Overall picks per hour: 152.4
Time values for non-regular users: 0
Zero-second picks: 98


In [39]:
# Breaks should cluster at fixed times if they are scheduled breaks
breaks = df[df['is_after_break'] & df['is_regular_picker']]
breaks['finish_time'].dt.hour.value_counts().sort_index()

# breaks for regular pickerd are 675, mostly at 23:00 and 01:00, some at 03:00.

finish_time
0       6
1     280
2       4
3      83
21      3
22     18
23    281
Name: count, dtype: int64

In [40]:
# Add to Cleaning Log
cleaning_log.append({'step': '7. Time per pick', 'rows': len(df),
                     'note': 'Finish-to-finish within shift; gaps of 15+ min are breaks; regular pickers only'})
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."
4,5. Merge lines into picks,206094,"11,142 lines merged; cases and volume conserved"
5,6. Analysis fields and flags,206094,"Added aisle, bay, picker flags and segments, s..."
6,7. Time per pick,206094,Finish-to-finish within shift; gaps of 15+ min...


## 8. Validate the clean table

A final check before export: the clean table must have the same total cases and volume as the source data (without the placeholder colli), with no duplicate picks and no impossible values.

In [41]:
# Compare with the raw data (without the placeholder colli) to prove nothing was lost
source = raw_df[raw_df['collli_id'] != 'colli_6599']

print('Picks:', len(df))
print('Cases - clean:', df['quantity'].sum(), '| source:', source['wm_vsola'].sum())
print('Volume - clean:', round(df['volume'].sum(), 3), '| source:', round(source['volume'].sum(), 3))
print('Duplicate picks (should be 0):', df.duplicated(['colli_id', 'product_id']).sum())
print('Negative volume (should be 0):', (df['volume'] < 0).sum())

Picks: 206094
Cases - clean: 1945575 | source: 1945575.0
Volume - clean: 4373.205 | source: 4373.205
Duplicate picks (should be 0): 0
Negative volume (should be 0): 0


In [42]:
# Only start_time, gap_sec and time_per_pick_sec should have missing values
df.isna().sum()[lambda s: s > 0]

start_time             85
gap_sec               361
time_per_pick_sec    1093
dtype: int64

In [43]:
cleaning_log.append({'step': '8. Validation', 'rows': len(df), 'note': 'Totals match the source; no duplicates or negative volume'})
pd.DataFrame(cleaning_log)

,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."
4,5. Merge lines into picks,206094,"11,142 lines merged; cases and volume conserved"
5,6. Analysis fields and flags,206094,"Added aisle, bay, picker flags and segments, s..."
6,7. Time per pick,206094,Finish-to-finish within shift; gaps of 15+ min...
7,8. Validation,206094,Totals match the source; no duplicates or nega...


# 9. Export
The validated pick-level table is saved as CSV and loaded into PostgreSQL (staging layer) for the SQL analysis and Power BI.

9.1 Order the columns

In [44]:
# Group columns by purpose so the table is easy to read in SQL and Power BI:
# identifiers, time, measures, then flags
column_order = [
    # identifiers
    'colli_id', 'product_id', 'picker', 'location', 'aisle', 'bay',
    # time
    'shift_date', 'shift_weekday', 'shift_week', 'first_finish_time', 'finish_time', 'start_time',
    # measures
    'quantity', 'volume', 'n_lines', 'gap_sec', 'time_per_pick_sec',
    # flags
    'holiday_period', 'partial_shift', 'is_regular_picker', 'picker_segment',
    'has_continuation', 'has_repeat', 'is_after_break',
    'is_single_line', 'is_bulk_line', 'is_new_product', 'is_one_shift_product',
]

# Make sure no column is lost or forgotten when reordering
assert set(column_order) == set(df.columns), set(df.columns) ^ set(column_order)  # assert stops the notebook with the set of missing or extra columns if they don't match

df = df[column_order]
print(df.shape)

(206094, 28)


9.2 Save the clean table and the cleaning summary

In [45]:

df.to_csv('../../data/picks_clean.csv', index=False)

cleaning_summary = pd.DataFrame(cleaning_log)
cleaning_summary.to_csv('../../data/cleaning_summary.csv', index=False)

print('Saved picks_clean.csv and cleaning_summary.csv in the data folder')
cleaning_summary

Saved picks_clean.csv and cleaning_summary.csv in the data folder


,step,rows,note
0,1. Load and prepare columns,217963,"Dropped 7 columns, renamed, fixed types"
1,2. Exclude placeholder colli,217236,Removed colli_6599 (727 rows)
2,3. Zero volume to NaN,217236,7 zero volumes set to missing
3,4. Shift calendar,217236,"Added shift_date, weekday, week, holiday and p..."
4,5. Merge lines into picks,206094,"11,142 lines merged; cases and volume conserved"
5,6. Analysis fields and flags,206094,"Added aisle, bay, picker flags and segments, s..."
6,7. Time per pick,206094,Finish-to-finish within shift; gaps of 15+ min...
7,8. Validation,206094,Totals match the source; no duplicates or nega...


### 9.3 Load into PostgreSQL

The validated table is loaded into the `staging` schema as `staging.picks_clean`. Connection details are read from the `.env` file, so no passwords are stored in the notebook.

In [46]:
#install packages
# %pip install sqlalchemy psycopg2-binary pandas python-dotenv


import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load .env from the project root, not from the notebook folder
load_dotenv(Path("../../.env"))

True

In [47]:
# connect jupyter to PostgreSQL
from sqlalchemy import create_engine

# Databse connection details
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
database = os.getenv("PG_DATABASE")

# Create PostgreSQL engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# test connection
with engine.connect() as conn:
    print("Connected to PostgreSQL!")

Connected to PostgreSQL!


In [51]:
# Read the cleaned data which exported earlier with correct data types
df_load = pd.read_csv(
    "../../data/picks_clean.csv",
    parse_dates=["shift_date", "first_finish_time", "finish_time", "start_time"],
    dtype={"gap_sec": "Int64", "time_per_pick_sec": "Int64"},
)


# Insert into the table created by 00_setup.sql (steps 1-2).
# 'append' keeps that table's data types, keys and rules; 'replace' would delete them.
df_load.to_sql(
    name="picks_clean",
    con=engine,
    schema="staging",
    if_exists="append",
    index=False,
    chunksize=10000,
)
# Ensure schema exists
with engine.connect() as conn:
    print("Rows in staging.picks_clean:", conn.execute(text("SELECT COUNT(*) FROM staging.picks_clean")).scalar())


Rows in staging.picks_clean: 206094
